In [ ]:
# -*- coding: utf-8 -*-
"""
Autoencoder físico guiado para compensação térmica de curvas de impedância.
Autor: Luiz Eduardo Abdala José (protótipo de IC SHM)
Descrição:
 - Mantém formato original das curvas
 - Aprende transformações físicas (offset, ganho, slope, centróide, energia, entropia, skew, etc.)
 - Treinado com base em curvas a várias temperaturas
"""

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ======================================================
# ======== 1. CARREGAR E PRÉ-PROCESSAR OS DADOS ========
# ======================================================

PKL_TREINO = "base_treino.pkl"
REF_TEMP = 20

base_tr = pd.read_pickle(PKL_TREINO)
f_cols = [c for c in base_tr.columns if c.startswith("f_")]
freqs = np.array([float(c[2:-2]) for c in f_cols])

X = base_tr[f_cols].to_numpy(float)
T = base_tr["temp_c"].to_numpy(float)
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-9)

# Normaliza temperatura entre 0 e 1
Tn = (T - T.min()) / (T.max() - T.min())

# Define curva de referência (média a REF_TEMP)
y_ref = np.median(base_tr.loc[base_tr["temp_c"] == REF_TEMP, f_cols].to_numpy(float), axis=0)
y_ref = (y_ref - y_ref.mean()) / (y_ref.std() + 1e-9)
Y_ref = np.repeat(y_ref[None, :], len(X), axis=0)

X_train = torch.tensor(X, dtype=torch.float32)
Y_ref_t = torch.tensor(Y_ref, dtype=torch.float32)
T_train = torch.tensor(Tn, dtype=torch.float32).unsqueeze(1)

input_dim = X.shape[1]

# ======================================================
# ======== 2. ARQUITETURA DO AUTOENCODER FÍSICO ========
# ======================================================

class PhysicsGuidedAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + 1, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)  # parâmetros físicos aprendidos
        )
        # nomes para fins de análise posterior
        self.param_names = [
            "offset", "gain", "slope", "shift",
            "skew_adj", "kurt_adj", "entropy_adj",
            "band_energy1", "band_energy2",
            "band_energy3", "band_energy4", "band_energy5",
            "roughness_adj", "centroid_adj", "amp_adj", "std_adj"
        ]

    def forward(self, x, t):
        # Entrada: curva + temperatura normalizada
        z = self.encoder(torch.cat([x, t], dim=1))

        # Extrai parâmetros físicos
        offset, gain, slope, shift = z[:, 0], z[:, 1], z[:, 2], z[:, 3]
        skew_adj, kurt_adj, entropy_adj = z[:, 4], z[:, 5], z[:, 6]
        bandE = z[:, 7:12]
        rough_adj, centroid_adj, amp_adj, std_adj = z[:, 12], z[:, 13], z[:, 14], z[:, 15]

        # ======= Decodificação física =======
        u = torch.linspace(-0.5, 0.5, x.shape[1], device=x.device).unsqueeze(0)
        x_c = x.clone()

        # offset
        x_c = x_c + offset.unsqueeze(1)

        # ganho
        x_mean = x_c.mean(dim=1, keepdims=True)
        x_c = x_mean + gain.unsqueeze(1) * (x_c - x_mean)

        # inclinação
        x_c = x_c + slope.unsqueeze(1) * u

        # deslocamento espectral (shift por amostra individual)
        f_shift = torch.zeros_like(x_c)
        for i in range(x_c.size(0)):
            s = int((shift[i] * 10).item())  # fator de escala do deslocamento
            f_shift[i] = torch.roll(x_c[i], shifts=s, dims=0)

        # pequenas correções adicionais
        x_c = f_shift + skew_adj.unsqueeze(1) * u**2 + kurt_adj.unsqueeze(1) * u**4
        x_c = x_c + entropy_adj.unsqueeze(1) * torch.sin(np.pi * u)
        x_c = x_c + 0.1 * (bandE.mean(dim=1, keepdim=True)) * u
        x_c = x_c + rough_adj.unsqueeze(1) * torch.cos(2 * np.pi * u)
        x_c = x_c + centroid_adj.unsqueeze(1) * u
        x_c = x_c * (1 + amp_adj.unsqueeze(1)) + std_adj.unsqueeze(1) * 0.01

        # normalização final para estabilidade
        x_c = (x_c - x_c.mean(dim=1, keepdims=True)) / (x_c.std(dim=1, keepdims=True) + 1e-9)

        return x_c, z

# ======================================================
# ================ 3. TREINAMENTO =======================
# ======================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PhysicsGuidedAutoencoder(input_dim=input_dim, latent_dim=16).to(device)
X_train, T_train, Y_ref_t = X_train.to(device), T_train.to(device), Y_ref_t.to(device)

opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

epochs = 500
loss_hist = []

for epoch in range(epochs):
    opt.zero_grad()
    recon, z = model(X_train, T_train)
    loss_recon = loss_fn(recon, Y_ref_t)
    loss_reg = 1e-4 * torch.mean(z**2)
    loss = loss_recon + loss_reg
    loss.backward()
    opt.step()
    loss_hist.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | Loss = {loss.item():.5f}")

# ======================================================
# ================ 4. VISUALIZAÇÃO ======================
# ======================================================

# Curva de perda
plt.figure(figsize=(6,4))
plt.plot(loss_hist)
plt.title("Evolução da perda (reconstrução + regularização)")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

# Obter parâmetros aprendidos
with torch.no_grad():
    _, z = model(X_train, T_train)
z = z.cpu().numpy()

plt.figure(figsize=(8,5))
for i, name in enumerate(model.param_names[:8]):
    plt.plot(T, z[:, i], '.', label=name)
plt.legend()
plt.title("Parâmetros físicos aprendidos vs Temperatura")
plt.xlabel("Temperatura (°C)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Exemplo de reconstrução de curvas
idx = np.argmin(np.abs(T - 70))  # exemplo em 70 °C
with torch.no_grad():
    recon, _ = model(X_train[idx:idx+1], T_train[idx:idx+1])
recon = recon.cpu().numpy().ravel()

plt.figure(figsize=(7,4))
plt.plot(freqs/1e3, X[idx], label=f"Original {T[idx]:.0f}°C", alpha=0.6, color='tab:red')
plt.plot(freqs/1e3, recon, label="Compensada (Autoencoder Físico)", color='tab:blue')
plt.plot(freqs/1e3, y_ref, '--', label="Referência 20°C", color='black')
plt.title("Compensação térmica física via autoencoder")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Magnitude normalizada")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()


In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu